In [ ]:
# 1
from pathlib import Path
import pyarrow
import shutil
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
import dill, json
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from catboost import CatBoostClassifier, Pool
import re
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_curve,
    confusion_matrix, average_precision_score, auc, roc_auc_score,
    classification_report
)
from src.predict import build_sessions_with_target

In [ ]:
# 2

# загружает данные sessions и hits, смотрит их размерность, первые строки, пропуски и дубликаты
sessions = pd.read_pickle("../data/raw/sample_sessions.pkl")
hits = pd.read_pickle("../data/raw/sample_hits.pkl")

print("Sessions shape:", sessions.shape)
print("Hits shape:", hits.shape)

# Первичный просмотр
display(sessions.head())
display(hits.head())

# Информация о таблицах
print(sessions.info())
print(hits.info())

# Проверка пропусков
print("Sessions nulls:\n", sessions.isna().sum())
print("Hits nulls:\n", hits.isna().sum())

# Проверка дубликатов
print("Sessions duplicates:", sessions.duplicated().sum())
print("Hits duplicates:", hits.duplicated().sum())


In [ ]:
# === Helpers for CatBoost pools (глобально) ===

MISSING_TOKEN = "__missing__"

def make_cb_pool(df, y=None):
    import pandas as pd, numpy as np
    num = df.select_dtypes(include=[np.number, "bool"]).columns
    cat = df.columns.difference(num)
    X = df.copy()
    for c in cat:
        X[c] = X[c].astype("object").fillna(MISSING_TOKEN).astype(str)
    cat_idx = list(X.columns.get_indexer(cat))
    return Pool(X, label=y, cat_features=cat_idx if cat_idx else None)

In [ ]:
import sys; print(sys.executable)

# 2.1. Описание атрибутов

## GA Sessions (sample_sessions.pkl)
- **session_id** — ID визита
- **client_id** — ID посетителя
- **visit_date** — дата визита
- **visit_time** — время визита
- **visit_number** — порядковый номер визита клиента
- **utm_source** — канал привлечения
- **utm_medium** — тип привлечения
- **utm_campaign** — рекламная кампания
- **utm_keyword** — ключевое слово
- **device_category** — тип устройства
- **device_os** — ОС устройства
- **device_brand** — марка устройства
- **device_model** — модель устройства
- **device_screen_resolution** — разрешение экрана
- **device_browser** — браузер
- **geo_country** — страна
- **geo_city** — город

## GA Hits (sample_hits.pkl)
- **session_id** — ID визита
- **hit_date** — дата события
- **hit_time** — время события
- **hit_number** — порядковый номер события в рамках сессии
- **hit_type** — тип события
- **hit_referer** — источник события
- **hit_page_path** — страница события
- **event_category** — тип действия
- **event_action** — действие
- **event_label** — тег действия
- **event_value** — значение результата действия

# 2.2. Интерпретация атрибутов в бизнес-контексте

- **UTM-метки (utm_source, utm_medium, utm_campaign, utm_keyword)**
  Используются для понимания источников и эффективности рекламных кампаний.

- **Device (category, os, brand, model, browser, screen_resolution)**
  Описывают устройство пользователя. Важно для анализа различий поведения мобильных и десктопных пользователей.

- **Geo (country, city)**
  Позволяет сегментировать пользователей по регионам. Например, сравнить конверсию из Москвы и регионов.

- **Visit_number**
  Показывает, какой по счёту визит совершает пользователь. Полезно для анализа лояльности и накопительного эффекта.

- **Hits (event_action, event_category)**
  Описывают конкретные действия пользователя на сайте. Именно из них строится целевая переменная (совершил ли пользователь целевое действие).

- **Target**
  Признак =1, если в сессии было хотя бы одно целевое действие (заявка, звонок и т.п.).


In [ ]:
# 2.3
# Диагностика: пропуски, пустые строки, редкие категории, проблемы с датами

def summarize_columns(df, cols):
    rows = []
    for c in cols:
        s = df[c]
        if s.dtype == "O":
            empty = (s.astype(str).str.strip() == "").sum()
        else:
            empty = 0
        rows.append({
            "column": c,
            "dtype": str(s.dtype),
            "n": len(s),
            "nan": s.isna().sum(),
            "empty_strings": empty,
            "unique": s.nunique(dropna=True),
            "top": s.value_counts(dropna=True).head(1).index.tolist()[0] if s.nunique(dropna=True) else None,
            "top_freq": int(s.value_counts(dropna=True).head(1).values[0]) if s.nunique(dropna=True) else 0,
        })
    return pd.DataFrame(rows).sort_values("nan", ascending=False)

key_cat_cols = [
    "utm_source","utm_medium","utm_campaign","utm_keyword",
    "device_category","geo_country","geo_city"
]

summary = summarize_columns(sessions, key_cat_cols + ["visit_date", "visit_time", "visit_number"])
display(summary)

# Проверка парсинга дат (проблемные значения)
visit_date_parsed = pd.to_datetime(sessions["visit_date"], errors="coerce", format=None)
bad_dates = sessions["visit_date"][visit_date_parsed.isna()].value_counts()

# Порог для редких категорий (макс(50, 0.5% от выборки))
rare_threshold = max(50, int(0.005 * len(sessions)))

rare_report = {}
for c in key_cat_cols:
    vc = sessions[c].astype(str).str.strip().str.lower().value_counts(dropna=False)
    rare_report[c] = vc[vc < rare_threshold]

print(f"rare_threshold = {rare_threshold}")
print("bad visit_date examples (top 10):")
display(bad_dates.head(10))

print("rare categories (первые 10 на столбец):")
for c, vc in rare_report.items():
    print(f"\n[{c}] rare count: {len(vc)}")
    display(vc.head(10))

In [ ]:
# 2.4
# Приведение к нормальному виду: очистка строк, даты, редкие категории, фичи даты

sessions_clean = sessions.copy()

# 1) Трим и нормализация строковых столбцов
obj_cols = sessions_clean.select_dtypes(include=["object"]).columns.tolist()
for c in obj_cols:
    sessions_clean[c] = sessions_clean[c].astype(str).str.strip()

# Нормализация UTM/категориальных к lower-case
for c in ["utm_source","utm_medium","utm_campaign","utm_keyword","device_category","geo_country","geo_city"]:
    if c in sessions_clean.columns:
        sessions_clean[c] = sessions_clean[c].str.lower()

# Пустые строки → NaN
for c in obj_cols:
    sessions_clean.loc[sessions_clean[c].str.len() == 0, c] = pd.NA

# 2) Даты и производные признаки
sessions_clean["visit_date_dt"] = pd.to_datetime(sessions_clean["visit_date"], errors="coerce")
sessions_clean["visit_weekday"] = sessions_clean["visit_date_dt"].dt.weekday  # 0=Mon..6=Sun
sessions_clean["visit_month"] = sessions_clean["visit_date_dt"].dt.month

# 3) Обработка редких категорий
def collapse_rare(s, threshold):
    vc = s.value_counts(dropna=False)
    rare_vals = vc[vc < threshold].index
    return s.where(~s.isin(rare_vals), other="other")

rare_threshold = max(50, int(0.005 * len(sessions_clean)))
for c in ["utm_source","utm_medium","utm_campaign","utm_keyword","device_category","geo_country","geo_city"]:
    if c in sessions_clean.columns:
        sessions_clean[c] = collapse_rare(sessions_clean[c], rare_threshold)

# 4) Числовые: visit_number (клип верхнего хвоста)
if "visit_number" in sessions_clean.columns:
    if not pd.api.types.is_numeric_dtype(sessions_clean["visit_number"]):
        sessions_clean["visit_number"] = pd.to_numeric(sessions_clean["visit_number"], errors="coerce")
    q99 = sessions_clean["visit_number"].quantile(0.99)
    sessions_clean["visit_number"] = sessions_clean["visit_number"].clip(lower=1, upper=q99)

# 5) Мини-очистка hits для target (трим/нижний регистр event_action)
hits_clean = hits.copy()
if "event_action" in hits_clean.columns:
    hits_clean["event_action"] = hits_clean["event_action"].astype(str).str.strip().str.lower()

print("sessions_clean shape:", sessions_clean.shape)
print("hits_clean shape:", hits_clean.shape)

In [ ]:
# 2.4.1
# Нормализация ключей для корректного merge: session_id -> str().strip() в обеих таблицах

sessions_clean["session_id"] = sessions_clean["session_id"].astype(str).str.strip()
hits_clean["session_id"]     = hits_clean["session_id"].astype(str).str.strip()
print("Нормализовали session_id в sessions_clean и hits_clean")


In [ ]:
# 3.0
# Диагностика: почему target почти/совсем пустой?

print("Типы session_id:", sessions_clean["session_id"].dtype, hits_clean["session_id"].dtype)

# Сколько разных session_id и их пересечение
sess_ids = set(sessions_clean["session_id"].astype(str))
hits_ids = set(hits_clean["session_id"].astype(str))
inter_size = len(sess_ids & hits_ids)
print(f"session_id: sessions={len(sess_ids)}, hits={len(hits_ids)}, intersection={inter_size}")

# Сколько целевых событий в hits_clean до merge
goal_actions = {
    'sub_car_claim_click','sub_car_claim_submit_click','sub_open_dialog_click',
    'sub_custom_question_submit_click','sub_call_number_click','sub_callback_submit_click',
    'sub_submit_success','sub_car_request_submit_click'
}
ea_sample = hits_clean["event_action"].astype(str).str.lower().value_counts().head(15)
print("\nTOP event_action (lower, top15):")
display(ea_sample)

is_goal_series = hits_clean["event_action"].astype(str).str.lower().isin(goal_actions)
print("is_goal True count (по hits_clean):", int(is_goal_series.sum()))
print("is_goal unique sessions:", hits_clean.loc[is_goal_series, "session_id"].astype(str).nunique())

# Построим sessions_with_target (1 — есть хотя бы одно целевое событие в сессии)

# нормализация ключей
sc = sessions_clean.copy()
hc = hits_clean.copy()
sc["session_id"] = sc["session_id"].astype(str).str.strip()
hc["session_id"] = hc["session_id"].astype(str).str.strip()

# отметим целевые события в hits_clean
hc["target"] = hc["event_action"].astype(str).str.lower().isin(goal_actions).astype(int)

# агрегируем до уровня session_id
targets_by_session = (
    hc.groupby("session_id", as_index=False)["target"].max()
)

# финальный датафрейм
sessions_with_target = (
    sc.merge(targets_by_session, on="session_id", how="left")
      .assign(target=lambda df: df["target"].fillna(0).astype(int))
)

print("\n[3.0] sessions_with_target:", sessions_with_target.shape)
display(sessions_with_target["target"].value_counts(dropna=False))


In [ ]:
# 3.0.1
# Проверка: сколько target=1 и какие вообще есть event_action

print("y (после ячейки 3) value_counts():")
display(sessions_with_target["target"].value_counts(dropna=False))

print("\nTOP event_action (lower) — top 50:")
top_actions = (
    hits_clean["event_action"]
    .astype(str).str.strip().str.lower()
    .value_counts()
    .head(50)
)
display(top_actions)


In [ ]:
# 3.0.0.1 — Разделение полного датасета на train/test с учётом дисбаланса
# [LEGACY] Этот сплит оставлен для отладки. Основной сплит выполняется в 11.1.



# X = sessions_with_target.drop(columns=["target"])
# y = sessions_with_target["target"]
#
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, stratify=y, random_state=42
# )
#
# print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
# print("Train target distribution:\n", y_train.value_counts())
# print("Test target distribution:\n", y_test.value_counts())

In [ ]:
# 3.0.2
# Авто-выделение кандидатов целевых событий по шаблонам + пересборка target


# 1) Порог и шаблоны (можно подправить)
min_freq = 5  # не брать совсем редкие
patterns = r"(submit|success|request|call|callback|claim|dialog|thank|thanks|send|apply|form|click)"

# 2) Список кандидатов из топ-таблицы
candidates = []
for action, cnt in top_actions.items():
    if cnt >= min_freq and re.search(patterns, action):
        candidates.append(action)

candidates = sorted(set(candidates))
print("Найдены кандидаты goal_actions:", candidates)

if len(candidates) == 0:
    print("⚠️ Кандидатов не найдено. Попробуй снизить min_freq или расширить patterns.")
else:
    # 3) Пересборка is_goal и target
    hits_tmp = hits_clean.copy()
    hits_tmp["event_action_lc"] = hits_tmp["event_action"].astype(str).str.strip().str.lower()
    hits_tmp["is_goal"] = hits_tmp["event_action_lc"].isin(candidates).astype("int8")

    goal_by_session = (
        hits_tmp.groupby("session_id", as_index=False)["is_goal"]
            .max()
            .rename(columns={"is_goal": "target"})
    )

    sessions_with_target = sessions_clean.merge(goal_by_session, on="session_id", how="left")
    sessions_with_target["target"] = sessions_with_target["target"].fillna(0).astype("int8")

    print("y value_counts() после пересборки target:")
    display(sessions_with_target["target"].value_counts())


In [ ]:
# 3.1
# Удаление ненужных атрибутов, которые не несут ценности для анализа/модели

drop_cols = ["client_id", "visit_time"]
sessions_with_target = sessions_with_target.drop(columns=[c for c in drop_cols if c in sessions_with_target.columns])

print("Оставшиеся колонки:", sessions_with_target.columns.tolist())

# 3.1m. Удаление ненужных атрибутов

- **client_id** — уникальный идентификатор пользователя.
  Для задач анализа и ML-модели не несёт полезной информации (слишком много уникальных значений, «ключ» вместо признака).

- **visit_time** — время визита в формате «часы:минуты:секунды».
  Удобнее работать с производными признаками (дата, день недели, месяц), чем с самим временем.

👉 Поэтому эти поля удалены.


In [ ]:
# 3.2
# Распределения числовых признаков (hist + boxplot)

num_cols = sessions_with_target.select_dtypes(include=["int64","float64"]).columns.tolist()
num_cols = [c for c in num_cols if c not in ["target"]]  # исключаем target

for col in num_cols:
    plt.figure(figsize=(10,4))

    plt.subplot(1,2,1)
    sns.histplot(sessions_with_target[col], bins=30, kde=True)
    plt.title(f"Hist {col}")

    plt.subplot(1,2,2)
    sns.boxplot(x=sessions_with_target[col])
    plt.title(f"Boxplot {col}")

    plt.show()


# 3.2m. Распределения числовых признаков

- **visit_number** — ожидаем правостороннее распределение: большинство визитов = первые, но есть и пользователи с большим числом визитов.
- Другие числовые (если есть, например event_value) также визуализированы.

👉 Boxplot помогает выявить выбросы, hist — понять форму распределения.


In [ ]:
# 3.3
# Корреляции числовых признаков (heatmap)

corr = sessions_with_target.corr(numeric_only=True)

plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Корреляция числовых признаков")
plt.show()


# 3.3m. Корреляции числовых признаков

- Корреляционная матрица показывает, есть ли сильные зависимости между числовыми признаками.
- Ожидаем, что между признаками слабая корреляция (разные сущности).
- Проверяем, чтобы не было полностью дублирующих фичей.

👉 Если признаки сильно коррелируют (>0.8), можно оставить один.


In [ ]:
# 3.4
# Связи категориальных и числовых признаков с target

# 1) Категориальные против target
cat_cols = ["utm_source","utm_medium","device_category","geo_country"]

for col in cat_cols:
    if col in sessions_with_target.columns:
        plt.figure(figsize=(8,4))
        sns.barplot(data=sessions_with_target, x=col, y="target")
        plt.title(f"CR в зависимости от {col}")
        plt.xticks(rotation=45)
        plt.show()

# 2) Числовые против target (violin)
for col in num_cols:
    plt.figure(figsize=(6,4))
    sns.violinplot(data=sessions_with_target, x="target", y=col)
    plt.title(f"{col} vs Target")
    plt.show()


# 3.4m. Связи категориальных и числовых признаков с целевой переменной

- Для категориальных (utm_source, utm_medium, device_category, geo_country) построен CR (среднее target).
  Это показывает, какие источники и устройства дают лучшую конверсию.

- Для числовых (например, visit_number) построен violinplot.
  Можно увидеть, как распределяется признак внутри классов target=0 и target=1.

👉 Эти графики помогают выявить потенциально сильные признаки для модели.


In [ ]:
# 4
# считает CR (конверсию) по категориям (utm_medium, utm_source, device_category).
def cr_table(df, by):
    g = df.groupby(by, dropna=False)["target"].agg(["count","mean"])\
          .rename(columns={"count":"visits","mean":"CR"})\
          .sort_values("visits", ascending=False)
    return g

cr_medium = cr_table(sessions_with_target, "utm_medium")
cr_source = cr_table(sessions_with_target, "utm_source")
cr_device = cr_table(sessions_with_target, "device_category")

display(cr_medium.head(10), cr_source.head(10), cr_device)

In [ ]:
# 5
# находит корень проекта, сохраняет подготовленный датасет sessions_with_target в правильную папку data/processed
# 1) Найти корень проекта: поднимаемся вверх, пока не найдём data/raw
cwd = Path.cwd().resolve()
ROOT = None
for p in [cwd] + list(cwd.parents):
    if (p / "data" / "raw").exists():
        ROOT = p
        break
assert ROOT is not None, f"Не нашёл корень: {cwd}"

# 2) Правильная папка для сохранения
PROC = ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)

# 3) Если файл уже лежит в notebooks/data/processed — переносим
wrong = cwd / "data" / "processed" / "sessions_with_target.parquet"
dst   = PROC / "sessions_with_target.parquet"
if wrong.exists() and not dst.exists():
    shutil.move(str(wrong), str(dst))
    print("Перенёс:", wrong, "→", dst)

# 4) Сохранить заново в правильное место (на случай обновлений)
sessions_with_target.to_parquet(dst, index=False)
print("Сохранил в:", dst)


In [ ]:
# 6
# строит график распределения целевой переменной (target).
# --- Целевая переменная ---
sessions_with_target["target"].value_counts(normalize=True).plot(
    kind="bar", title="Распределение целевой переменной (target)", rot=0
)
plt.show()


In [ ]:
# 7
# строит распределения категориальных признаков (UTM-метки, устройство, страна).
for col in ["utm_medium", "utm_source", "device_category", "geo_country"]:
    plt.figure(figsize=(8,4))
    sns.countplot(data=sessions_with_target, x=col, order=sessions_with_target[col].value_counts().index)
    plt.title(f"Распределение {col}")
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
# 8
# визуализирует CR (конверсию) по тем же категориальным признакам
def plot_cr(df, col):
    cr = df.groupby(col)["target"].mean().sort_values(ascending=False)
    cr.plot(kind="bar", figsize=(8,4), title=f"CR по {col}")
    plt.xticks(rotation=45)
    plt.ylabel("CR")
    plt.show()

for col in ["utm_medium", "utm_source", "device_category", "geo_country"]:
    plot_cr(sessions_with_target, col)

In [ ]:
# 9
# показывает распределение номера визита и как от него зависит конверсия.
# Распределение visit_number
plt.figure(figsize=(8,4))
sns.histplot(sessions_with_target["visit_number"], bins=20, kde=False)
plt.title("Распределение visit_number")
plt.show()

# CR в зависимости от номера визита
sns.barplot(data=sessions_with_target, x="visit_number", y="target")
plt.title("CR в зависимости от номера визита")
plt.show()

In [ ]:
# 10
# считает частоту целевых действий (event_action) в логах событий
goal_actions = {
    "sub_car_claim_click",
    "sub_car_claim_submit_click",
    "sub_open_dialog_click",
    "sub_custom_question_submit_click",
    "sub_call_number_click",
    "sub_callback_submit_click",
    "sub_submit_success",
    "sub_car_request_submit_click"
}

hits[hits["event_action"].isin(goal_actions)]["event_action"].value_counts()

In [ ]:
# 11
# Подготовка признаков X, y для ML-модели

# формируем y = target;
# убираем ненужные столбцы (session_id, visit_date, visit_date_dt);
# оставляем X только с признаками для модели;
# разделяем список категориальных и числовых признаков.

# Целевая переменная
y = sessions_with_target["target"]

# Исключаем технические поля
drop_cols = [
    "session_id",     # уникальный идентификатор сессии
    "visit_date",     # оставляем производные: weekday, month
    "visit_date_dt",   # datetime-объект, его напрямую не используем
    "target"
]

X = sessions_with_target.drop(columns=[c for c in drop_cols if c in sessions_with_target.columns])

# Отделяем категориальные и числовые
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
num_cols = X.select_dtypes(include=["int64","float64"]).columns.tolist()

print("Целевая переменная:", y.name)
print("Число объектов:", len(X))
print("Категориальные признаки:", cat_cols)
print("Числовые признаки:", num_cols)


In [ ]:
# 11.1
# Train/Test split + безопасная подготовка признаков (импьютеры + OHE с поддержкой пропусков)

# делим данные на train/test (стратификация по target чтобы сохранить баланс классов),
# строим пайплайн для подготовки:
# StandardScaler для числовых признаков,
# OneHotEncoder для категориальных,
# применяем трансформации и показываем итоговую размерность.



# Пересчитаем списки признаков на всякий случай (учитываем и 'object', и 'string')
cat_cols_11 = X.select_dtypes(include=["object", "string"]).columns.tolist()
num_cols_11 = X.select_dtypes(include=["number", "bool"]).columns.tolist()
num_cols_11 = [c for c in num_cols_11 if c != "target"]

# Приводим pd.NA -> np.nan в категориальных
for c in cat_cols_11:
    # оставляем как объектные строки, заменяем <NA> на np.nan
    X[c] = X[c].astype("object").replace({pd.NA: np.nan})

# Разделение
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Train size:", X_train.shape, " Test size:", X_test.shape)

# Пайплайны препроцессинга
num_pipeline = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler())
])
cat_pipeline = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=True))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", num_pipeline, num_cols_11),
    ("cat", cat_pipeline, cat_cols_11),
])

# Проверим трансформации
X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared  = preprocessor.transform(X_test)

print("После трансформации:",
      "X_train_prepared:", X_train_prepared.shape,
      "X_test_prepared:",  X_test_prepared.shape)



In [ ]:
# 11.1.1
# Безопасный split: гарантируем >=1 объекта каждого класса в train и test



pos = int((y == 1).sum())
neg = int((y == 0).sum())
n = len(y)

print(f"Всего: {n}, pos: {pos}, neg: {neg}")

if pos < 2:
    raise ValueError("target=1 встречается < 2 раз — разрез невозможен. Проверь логику формирования target (ячейка 3).")

# подбираем test_size, чтобы в test и train был хотя бы 1 положительный
def pick_test_size(pos, base=0.2):
    ts = base
    for ts in [0.2, 0.25, 0.3, 0.33, 0.4, 0.5]:
        test_pos = max(1, int(round(pos * ts)))
        train_pos = pos - test_pos
        if train_pos >= 1 and test_pos >= 1:
            return ts
    return 0.5  # fallback

ts = pick_test_size(pos)
print(f"Используем test_size={ts}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=ts, random_state=42, stratify=y
)

print("Train counts:\n", y_train.value_counts())
print("Test  counts:\n", y_test.value_counts())


In [ ]:
# 11.2
# Baseline модель: Logistic Regression

# обучаем логистическую регрессию как baseline,
# считаем ROC-AUC для train/test,
# выводим classification report и confusion matrix.

# Модель с пайплайном (включаем препроцессинг)
log_reg_clf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("clf", LogisticRegression(solver="saga", max_iter=1000,
                           class_weight=None, random_state=42, n_jobs=-1))
])

# Обучение
log_reg_clf.fit(X_train, y_train)

# Предсказания вероятностей для ROC-AUC
y_train_proba = log_reg_clf.predict_proba(X_train)[:, 1]
y_test_proba = log_reg_clf.predict_proba(X_test)[:, 1]

# Метрика ROC-AUC
roc_auc_train = roc_auc_score(y_train, y_train_proba)
roc_auc_test = roc_auc_score(y_test, y_test_proba)

print(f"ROC-AUC train: {roc_auc_train:.3f}")
print(f"ROC-AUC test: {roc_auc_test:.3f}")

# Классификационный отчёт
y_pred = log_reg_clf.predict(X_test)
print("\nClassification report (test):\n", classification_report(y_test, y_pred))
print("Confusion matrix (test):\n", confusion_matrix(y_test, y_pred))


In [ ]:
# 11.3
# Модели на деревьях: RandomForest и GradientBoosting

# обучаем RandomForest и GradientBoosting,
# считаем ROC-AUC для train/test,
# сохраняем результаты в словарь results.

models = {
    "RandomForest": RandomForestClassifier(
        n_estimators=200, max_depth=None, random_state=42, n_jobs=-1
    ),
    "GradientBoosting": GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=3, random_state=42
    )
}

results = {}

for name, model in models.items():
    clf = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("clf", model)
    ])

    clf.fit(X_train, y_train)
    y_train_proba = clf.predict_proba(X_train)[:, 1]
    y_test_proba = clf.predict_proba(X_test)[:, 1]

    roc_auc_train = roc_auc_score(y_train, y_train_proba)
    roc_auc_test = roc_auc_score(y_test, y_test_proba)

    results[name] = (roc_auc_train, roc_auc_test)

    print(f"\n{name}")
    print(f"ROC-AUC train: {roc_auc_train:.3f}")
    print(f"ROC-AUC test: {roc_auc_test:.3f}")

results


In [ ]:
# 11.4
# Сравнение моделей: ROC-AUC + ROC-кривые

# сравнивает LogReg, RandomForest, GradientBoosting на одном графике ROC,
# выводит таблицу с ROC-AUC по каждой модели.
# Сохраняем модели для сравнения

final_models = {
    "LogisticRegression": log_reg_clf,
    "RandomForest": Pipeline(steps=[("preprocessor", preprocessor), ("clf", models["RandomForest"])]),
    "GradientBoosting": Pipeline(steps=[("preprocessor", preprocessor), ("clf", models["GradientBoosting"])]),
}

roc_results = []

plt.figure(figsize=(8,6))

for name, clf in final_models.items():
    clf.fit(X_train, y_train)
    y_test_proba = clf.predict_proba(X_test)[:, 1]

    fpr, tpr, _ = roc_curve(y_test, y_test_proba)
    roc_auc = auc(fpr, tpr)

    plt.plot(fpr, tpr, label=f"{name} (AUC = {roc_auc:.3f})")
    roc_results.append({"Model": name, "ROC-AUC": roc_auc})

# ROC-plot
plt.plot([0,1],[0,1],"k--", label="Random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC-кривые моделей")
plt.legend(loc="lower right")
plt.show()

# Таблица результатов
roc_table = pd.DataFrame(roc_results)
display(roc_table.sort_values("ROC-AUC", ascending=False))


In [ ]:
# 11.5
# Сохранение лучшей модели через dill

# модель выбирается автоматически по результатам 11.4,
# сохраняется с именем, где указан её тип (best_model_GradientBoosting.pkl, например),
# в выводе сразу видно, какая модель лучшая и с каким ROC-AUC.

# Определяем модель с наибольшим ROC-AUC из таблицы 11.4
best_row = roc_table.sort_values("ROC-AUC", ascending=False).iloc[0]
best_name = best_row["Model"]
best_score = best_row["ROC-AUC"]

best_model = final_models[best_name]

# создаём папку models если её нет
models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

model_path = models_dir / f"best_model_{best_name}.pkl"

with open(model_path, "wb") as f:
    dill.dump(best_model, f)

print(f"Лучшая модель: {best_name} (ROC-AUC={best_score:.3f})")
print("Сохранена в:", model_path)


### Вывод по 11.7
- PR-AUC (AP) = 0.219 → качество умеренное, но выше случайного.
- Лучший по F1: threshold ≈ 0.16 → precision ≈ 0.22, recall ≈ 0.46, f1 ≈ 0.30.
- Лучший по Youden J: threshold ≈ 0.10 → recall ≈ 0.79, но много FP.
- При precision ≥0.3 лучший порог ≈ 0.20 → precision ≈ 0.30, recall ≈ 0.19.

**Рекомендуемый рабочий порог: 0.16 (по F1).**
Окончательный выбор зависит от бизнес-приоритета:
- если важен recall (поймать максимум лидов) — порог 0.10;
- если важен баланс — порог 0.16;
- если нужен контроль за качеством лидов (precision ≥0.3) — порог 0.20.


In [ ]:
# 11.8 — LogisticRegression (balanced) с препроцессингом для категориальных

# 1) Разделим признаки на числовые и категориальные
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.columns.difference(num_cols).tolist()

# 2) Препроцессинг
num_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler(with_mean=False))  # with_mean=False для совместимости со sparse
])

cat_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore"))  # sparse выход — ок для solver='saga'
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols),
    ],
    remainder="drop"
)

# 3) Модель
log_reg_bal = Pipeline(steps=[
    ("pre", preprocess),
    ("clf", LogisticRegression(
        class_weight="balanced",
        solver="saga",        # поддерживает sparse
        max_iter=1000,
        random_state=42
    ))
])

display("Колонок в X_train:", X_train.shape[1])
display("Есть ли 'target' в X_train?:", "target" in X_train.columns)
display("Примеры колонок:", X_train.columns[:40].tolist())

# 4) Обучение и оценка
log_reg_bal.fit(X_train, y_train)

y_train_proba = log_reg_bal.predict_proba(X_train)[:, 1]
y_test_proba  = log_reg_bal.predict_proba(X_test)[:, 1]

display("LogisticRegression (balanced + OHE)")
display(f"ROC-AUC train: {roc_auc_score(y_train, y_train_proba):.3f}")
display(f"ROC-AUC test:  {roc_auc_score(y_test,  y_test_proba):.3f}")

y_test_pred = (y_test_proba >= 0.5).astype(int)
display("\nClassification report (test) @0.5:")
display(classification_report(y_test, y_test_pred, digits=3))
display("Confusion matrix (test) @0.5:")
display(confusion_matrix(y_test, y_test_pred))



In [ ]:
# 11.9 — GradientBoosting (balanced via sample_weight) с препроцессингом

from sklearn.utils.class_weight import compute_sample_weight

# 1) Разделяем фичи
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.columns.difference(num_cols).tolist()

# 2) Препроцессинг
num_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())  # dense
])

cat_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))  # dense для GBC
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols),
    ],
    remainder="drop"
)

# 3) Модель в пайплайне
gb_bal = Pipeline(steps=[
    ("pre", preprocess),
    ("clf", GradientBoostingClassifier(random_state=42))
])

# 4) Веса классов и обучение (важно: прокинуть как clf__sample_weight)
sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)
gb_bal.fit(X_train, y_train, clf__sample_weight=sample_weights)

# 5) Оценка
y_train_proba = gb_bal.predict_proba(X_train)[:, 1]
y_test_proba  = gb_bal.predict_proba(X_test)[:, 1]

display("GradientBoosting (balanced + OHE)")
display(f"ROC-AUC train: {roc_auc_score(y_train, y_train_proba):.3f}")
display(f"ROC-AUC test:  {roc_auc_score(y_test,  y_test_proba):.3f}")

y_test_pred = (y_test_proba >= 0.5).astype(int)
display("\nClassification report (test) @0.5:")
display(classification_report(y_test, y_test_pred, digits=3))
display("Confusion matrix (test) @0.5:")
display(confusion_matrix(y_test, y_test_pred))



In [ ]:
# 11.10 — CatBoostClassifier (через make_cb_pool, с учетом дисбаланса)

train_pool = make_cb_pool(X_train, y_train)
test_pool  = make_cb_pool(X_test,  y_test)

pos = int((y_train == 1).sum()); neg = int((y_train == 0).sum())
scale_pos_weight = neg / max(pos, 1)

cb = CatBoostClassifier(
    iterations=600,
    depth=6,
    learning_rate=0.08,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    scale_pos_weight=scale_pos_weight,
    verbose=100
)

cb.fit(train_pool, eval_set=test_pool, use_best_model=True)

y_train_proba = cb.predict_proba(train_pool)[:, 1]
y_test_proba  = cb.predict_proba(test_pool)[:, 1]

print("CatBoost (with make_cb_pool)")
print(f"ROC-AUC train: {roc_auc_score(y_train, y_train_proba):.3f}")
print(f"ROC-AUC test:  {roc_auc_score(y_test,  y_test_proba):.3f}")

y_test_pred = (y_test_proba >= 0.5).astype(int)
print("\nClassification report (test) @0.5:")
print(classification_report(y_test, y_test_pred, digits=3))
print("Confusion matrix (test) @0.5:")
print(confusion_matrix(y_test, y_test_pred))





In [ ]:
# 11.11 — Сравнение моделей (ROC-AUC, PR-AUC) с поддержкой CatBoost

def eval_model(name, model, X_tr, y_tr, X_te, y_te):
    if isinstance(model, CatBoostClassifier):
        tr_pool = make_cb_pool(X_tr)
        te_pool = make_cb_pool(X_te)
        ytr = model.predict_proba(tr_pool)[:, 1]
        yte = model.predict_proba(te_pool)[:, 1]
    else:
        ytr = model.predict_proba(X_tr)[:, 1]
        yte = model.predict_proba(X_te)[:, 1]
    return {
        "Model": name,
        "ROC-AUC train": roc_auc_score(y_tr, ytr),
        "ROC-AUC test":  roc_auc_score(y_te, yte),
        "PR-AUC test":   average_precision_score(y_te, yte)
    }

results = []
models = []
if "log_reg_bal" in globals(): models.append(("LogisticRegression (balanced)", log_reg_bal))
if "gb_bal"      in globals(): models.append(("GradientBoosting (balanced)", gb_bal))
if "cb"          in globals(): models.append(("CatBoost (spw)", cb))

for name, mdl in models:
    try:
        results.append(eval_model(name, mdl, X_train, y_train, X_test, y_test))
    except Exception as e:
        print(f"[WARN] {name}: {e}")

df_scores = pd.DataFrame(results).sort_values("ROC-AUC test", ascending=False).reset_index(drop=True)
display(df_scores)




In [ ]:
# 11.12 — Подбор порога для лучшей модели (CatBoost)


te_pool = make_cb_pool(X_test)
y_scores = cb.predict_proba(te_pool)[:, 1]
y_true   = y_test.values

ths = np.round(np.linspace(0.01, 0.99, 99), 2)
rows = []
for t in ths:
    y_pred = (y_scores >= t).astype(int)
    rows.append((t,
                 precision_score(y_true, y_pred, zero_division=0),
                 recall_score(y_true, y_pred, zero_division=0),
                 f1_score(y_true, y_pred, zero_division=0)))
thr_df = pd.DataFrame(rows, columns=["thr","precision","recall","f1"])
best_f1 = thr_df.sort_values("f1", ascending=False).iloc[0]

fpr, tpr, roc_ths = roc_curve(y_true, y_scores)
youden_thr = float(roc_ths[np.argmax(tpr - fpr)])

ap = average_precision_score(y_true, y_scores)
print(f"PR-AUC (AP): {ap:.3f}\n")
print("Лучший по F1:\n", best_f1, "\n")
print(f"Лучший по Youden J (ROC): thr ≈ {youden_thr:.3f}")
CHOSEN_THR = float(best_f1.thr)
print(f"\nРекомендуемый threshold: {CHOSEN_THR:.2f}")


In [ ]:
# 11.13 — save best CatBoost + meta

models_dir = Path("../models"); models_dir.mkdir(parents=True, exist_ok=True)
model_path = models_dir / "best_model_full.pkl"
meta_path  = models_dir / "best_model_full_meta.json"

with open(model_path,"wb") as f: dill.dump(cb, f)
meta = {
  "best_name": "CatBoost (spw)",
  "chosen_threshold": CHOSEN_THR,
  "roc_auc_test": float(roc_auc_score(y_test, cb.predict_proba(te_pool)[:,1])),
  "train_feature_names": list(X_train.columns)
}
with open(meta_path,"w",encoding="utf-8") as f: json.dump(meta,f,ensure_ascii=False,indent=2)
display(f"Сохранено:\n  {model_path}\n  {meta_path}\n{meta}")

# Контроль при выбранном пороге
y_pred = (y_scores >= CHOSEN_THR).astype(int)
print("\nClassification report (test) @thr:")
print(classification_report(y_true, y_pred, digits=3))
print("Confusion matrix (test) @thr:")
print(confusion_matrix(y_true, y_pred))



In [ ]:
# 11.14 — Retrain CatBoost on large sample from FULL data (same prep as in predict.py)

# 0) сборка swt_full
sessions = pd.read_pickle(Path("../data/raw/ga_sessions.pkl"))
hits = pd.read_pickle(Path("../data/raw/ga_hits.pkl"))
swt_full = build_sessions_with_target(sessions, hits)
del sessions, hits

# восстановим visit_weekday / visit_month, если их нет
if ("visit_weekday" not in swt_full.columns) or ("visit_month" not in swt_full.columns):
    if "visit_date" in swt_full.columns:
        dt = pd.to_datetime(swt_full["visit_date"], errors="coerce")
    elif "visit_time" in swt_full.columns:
        dt = pd.to_datetime(swt_full["visit_time"], unit="s", errors="coerce")
    else:
        dt = pd.Series(pd.NaT, index=swt_full.index)
    if "visit_weekday" not in swt_full.columns:
        swt_full["visit_weekday"] = dt.dt.weekday.astype("float32")
    if "visit_month" not in swt_full.columns:
        swt_full["visit_month"] = dt.dt.month.astype("float32")

print("[INFO] swt_full shape:", swt_full.shape)


def make_cb_pool_train(dfX: pd.DataFrame, y=None):
    num = dfX.select_dtypes(include=[np.number, "bool"]).columns
    cat = dfX.columns.difference(num)
    X = dfX.copy()
    if len(num) > 0:
        X[num] = X[num].astype("float32")
    for c in cat:
        s = X[c].astype("object").astype(str).str.strip().str.lower()
        s = s.replace({"nan": MISSING_TOKEN, "": MISSING_TOKEN})
        X[c] = s
    cat_idx = list(X.columns.get_indexer(cat))
    return Pool(X, label=y, cat_features=cat_idx if cat_idx else None)


# 1) фичи/таргет
drop_cols = [c for c in ["session_id", "client_id", "target"] if c in swt_full.columns]
X_all = swt_full.drop(columns=drop_cols)
y_all = swt_full["target"].astype("int8")

# 2) крупный стратифицированный семпл
SAMPLE_SIZE = 600_000
if len(X_all) > SAMPLE_SIZE:
    Xs, _, ys, _ = train_test_split(
        X_all, y_all,
        test_size=(len(X_all) - SAMPLE_SIZE) / len(X_all),
        stratify=y_all, random_state=42
    )
else:
    Xs, ys = X_all, y_all

# 3) train/valid split
Xtr, Xva, ytr, yva = train_test_split(Xs, ys, test_size=0.2, stratify=ys, random_state=42)

# 4) приведение типов как в predict
numeric_cols = {"visit_number", "visit_weekday", "visit_month"}
for df in (Xtr, Xva):
    for c in df.columns:
        if c in numeric_cols:
            df[c] = pd.to_numeric(df[c], errors="coerce").astype("float32")
        else:
            df[c] = df[c].astype("object")

# === Шаг 2: ручной Grid Search для CatBoost (совместимо с любой версией) ===

# баланс классов
pos = int((ytr == 1).sum())
neg = int((ytr == 0).sum())
scale_pos_weight = neg / max(pos, 1)
print(f"scale_pos_weight={scale_pos_weight:.3f} (pos={pos}, neg={neg})")

# пулы
tr_pool = make_cb_pool_train(Xtr, ytr)
va_pool = make_cb_pool_train(Xva, yva)

# 2.1 grid на уменьшенном трейне
GRID_SAMPLE = 300_000
if len(Xtr) > GRID_SAMPLE:
    Xtr_gs, _, ytr_gs, _ = train_test_split(
        Xtr, ytr,
        test_size=(len(Xtr) - GRID_SAMPLE) / len(Xtr),
        stratify=ytr, random_state=42
    )
else:
    Xtr_gs, ytr_gs = Xtr, ytr

tr_pool_gs = make_cb_pool_train(Xtr_gs, ytr_gs)
va_pool_gs = make_cb_pool_train(Xva, yva)

# базовые параметры и пространство поиска
from itertools import product

base_params = dict(
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    scale_pos_weight=scale_pos_weight,
    iterations=600,  # короче для грид-поиска
    early_stopping_rounds=50,
    verbose=100
)
param_space = {
    "depth": [6, 8],
    "learning_rate": [0.05, 0.07],
    "l2_leaf_reg": [3, 6],
}

print("[INFO] manual grid (8 combos) ...")
candidates = []
for depth, lr, l2 in product(param_space["depth"], param_space["learning_rate"], param_space["l2_leaf_reg"]):
    params = dict(base_params)
    params.update({"depth": depth, "learning_rate": lr, "l2_leaf_reg": l2})

    model_gs = CatBoostClassifier(**params)
    model_gs.fit(tr_pool_gs, eval_set=va_pool_gs, use_best_model=True)

    val_auc = None
    bs = getattr(model_gs, "best_score_", None)
    if isinstance(bs, dict):
        val_auc = (bs.get("validation") or bs.get("eval") or {}).get("AUC", None)
    if val_auc is None:
        p_va_gs = model_gs.predict_proba(va_pool_gs)[:, 1]
        val_auc = roc_auc_score(yva, p_va_gs)

    print(f"[grid] depth={depth}, lr={lr}, l2={l2} -> valid AUC={val_auc:.6f}")
    candidates.append((val_auc, {"depth": depth, "learning_rate": lr, "l2_leaf_reg": l2}))

best_params = max(candidates, key=lambda x: x[0])[1]
print("[INFO] grid best params:", best_params)

# 2.2 финальная модель с лучшими гиперпараметрами
final_params = dict(base_params)
final_params.update(best_params)
final_params.update({
    "iterations": 1000,
    "early_stopping_rounds": 100,
})
cb = CatBoostClassifier(**final_params)
cb.fit(tr_pool, eval_set=va_pool, use_best_model=True)

# 5) метрики
p_tr = cb.predict_proba(tr_pool)[:, 1]
p_va = cb.predict_proba(va_pool)[:, 1]
roc_tr = roc_auc_score(ytr, p_tr)
roc_va = roc_auc_score(yva, p_va)
pr_va = average_precision_score(yva, p_va)
print(f"ROC-AUC train: {roc_tr:.3f}")
print(f"ROC-AUC valid: {roc_va:.3f}")
print(f"PR-AUC  valid: {pr_va:.3f}")

# 6) подбор порога по F1
thr_grid = np.linspace(0.05, 0.80, 76)
best_thr, best_f1 = 0.5, -1.0
for thr in thr_grid:
    f1 = f1_score(yva, (p_va >= thr).astype("int8"))
    if f1 > best_f1:
        best_f1, best_thr = f1, thr
print(f"Best thr by F1 (valid): {best_thr:.3f}, F1={best_f1:.3f}")
print(classification_report(yva, (p_va >= best_thr).astype('int8'), digits=3))

# 7) сохранить модель и meta
models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)
model_path = models_dir / "best_model_full_retrained.pkl"
with open(model_path, "wb") as f:
    dill.dump(cb, f)

meta = {
    "best_name": "CatBoost (spw+grid)",
    "chosen_threshold": float(best_thr),
    "roc_auc_valid": float(roc_va),
    "pr_auc_valid": float(pr_va),
    "train_feature_names": list(Xtr.columns),
    "best_params": best_params
}
with open(models_dir / "best_model_full_retrained_meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print("Saved:", model_path.name, "and meta.")
